# Lab 8: The task model of automation
**MST 0441: Consumers, Trade and Business Strategy**
*Second in-person block, session 8. About 45 minutes.*

You are the firm. Production is a list of **tasks**, not one black box. For each
task you pick the cheaper supplier: a worker at wage $w$, or a machine that does
task $i$ with productivity $A_i$ at rental rate $r$.

$$\text{machine cost} = \frac{r}{A_i}, \qquad \text{worker cost} = w
\qquad\Longrightarrow\qquad \text{automate iff } A_i > \frac{r}{w} \equiv \bar{A}.$$

Everything today follows from that one inequality.

### How this lab works

1. Run `Runtime -> Run all` first. The notebook runs as it stands. Read the
   output, then return to the top.
2. Do the cells marked YOUR TURN. Each asks for a number, a line, or a short
   function. The cells are independent; a wrong answer in one does not affect
   the others.
3. Each YOUR TURN ends with a `check(...)` that reports whether your answer
   matches. Nothing raises an error.
4. The last section, *Work with your assistant*, calls your model from code
   through `ask_model()`. One-time setup: the key guide on It's Learning. No key?
   Every prompt is a plain string you can copy into a chat window instead;
   paste the reply where marked. Test each reply in code before accepting it.

Nothing to install. `numpy`, `matplotlib` and `requests` are preinstalled in
Colab.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from fractions import Fraction

def check(name, got, want, tol=1e-9):
    """Friendly checker: prints, never raises."""
    if got is None:
        print(f"  ..  {name}: not filled in yet")
        return False
    try:
        if isinstance(want, (list, tuple, np.ndarray)):
            good = np.allclose(np.asarray(got, dtype=float),
                               np.asarray(want, dtype=float), atol=tol)
        elif isinstance(want, str):
            good = str(got).strip().lower() == want.strip().lower()
        elif isinstance(want, bool):
            good = bool(got) is want
        else:
            good = abs(float(got) - float(want)) < tol
    except Exception as e:
        print(f"  XX  {name}: could not compare ({type(e).__name__}: {e})")
        return False
    print(f"  {'OK ' if good else 'XX '} {name} = {got}" + ("" if good else f"   (expected {want})"))
    return good

print("ready")

## 1. The model, in about ten lines  (given: just run it)

In [ ]:
def automation(A, chi, w, r):
    """Solve the firm's task-by-task automate-or-not problem.

    A   : machine productivity in each task (higher = easier to automate)
    chi : each task's share of total cost (should sum to 1)
    w   : wage
    r   : rental rate of the machine (the "price of AI")
    """
    A, chi = np.asarray(A, float), np.asarray(chi, float)

    Abar = r / w                       # the cutoff
    automated = A > Abar

    # Cost saving on task i, relative to doing it with labour:
    #     pi_i = 1 - (r/A_i)/w = 1 - Abar/A_i   if automated, else 0
    pi = np.where(automated, 1.0 - Abar / A, 0.0)

    # Hulten: the productivity gain is the share-weighted sum of the savings.
    dlnTFP = float(np.sum(chi * pi))
    return {"Abar": Abar, "automated": automated, "pi": pi, "dlnTFP": dlnTFP}


A   = [3.0, 1.5, 0.5]            # task 1 easiest to automate, task 3 hardest
chi = [1/3, 1/3, 1/3]            # equal cost shares
w, r = 1.0, 2.0                  # so Abar = 2.0

out = automation(A, chi, w, r)
print(f"cutoff Abar = r/w = {out['Abar']:.2f}")
for i, (a, auto, p) in enumerate(zip(A, out["automated"], out["pi"]), 1):
    print(f"  task {i}:  A = {a:>4.1f}  ->  {'MACHINE' if auto else 'worker '}"
          f"   saving pi = {p:6.1%}")
print(f"productivity gain = sum(chi_i * pi_i) = {out['dlnTFP']:.2%}")

check("gain", out["dlnTFP"], 1/9)

## 2. YOUR TURN: Acemoglu's actual numbers

He estimates that the tasks AI can currently do amount to $\chi = 4.6\%$ of GDP,
with an average cost saving of $\pi = 15.4\%$. Compute the gain without using
`automation()`: it is $\chi\pi$.

Then split the exposed tasks into easy ($\chi = 3.3\%$, $\pi = 15.4\%$) and hard
($\chi = 1.2\%$, $\pi = 4.0\%$) and recompute.

In [ ]:
gain_whole = None      # <-- YOUR TURN: chi * pi, as a decimal
gain_split = None      # <-- YOUR TURN: easy + hard

print(f"whole: {gain_whole}")
print(f"split: {gain_split}")
check("whole", gain_whole, 0.046 * 0.154)
check("split", gain_split, 0.033 * 0.154 + 0.012 * 0.040)

# In one line: why does splitting them LOWER the answer?

## 3. YOUR TURN: cost shares matter more than you expect

Change `chi` from `(1/3, 1/3, 1/3)` to `(0.6, 0.2, 0.2)`, leaving `A`, `w` and
`r` alone. The **same single task** is automated. Predict the new gain first.

In [ ]:
predicted = None       # <-- YOUR TURN: your prediction for the new gain

new_chi = [0.6, 0.2, 0.2]
out2 = automation(A, new_chi, w, r)
print(f"gain with equal shares : {out['dlnTFP']:.2%}")
print(f"gain with (0.6,0.2,0.2): {out2['dlnTFP']:.2%}")
check("your prediction", predicted, out2["dlnTFP"], tol=1e-4)

# General lesson in one sentence:

## 4. Cheaper AI, and why the curve is kinked  (given, then YOUR TURN)

In [ ]:
def sweep(A, chi, w, r_hi=3.0, r_lo=0.2, n=400):
    grid = np.linspace(r_hi, r_lo, n)
    gains = np.array([automation(A, chi, w, r)["dlnTFP"] for r in grid])
    return grid, gains

grid, gains = sweep(A, chi, w)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(grid, 100 * gains, lw=2)
ax.set_xlabel("price of AI, r   (falling to the right ->)")
ax.set_ylabel("productivity gain, %")
ax.invert_xaxis(); ax.grid(alpha=.3)
for i, a in enumerate(A, 1):
    ax.axvline(w * a, color="grey", ls=":", lw=1)
    ax.text(w * a, ax.get_ylim()[1]*.9, f" task {i} enters", rotation=90,
            va="top", fontsize=8, color="grey")
ax.set_title("Cheaper AI automates more tasks, in kinked steps")
plt.show()

In [ ]:
# <-- YOUR TURN: each kink is a task crossing the cutoff, at r = w * A_i.
# Predict where the kinks would be if A = [3.0, 2.9, 2.8], then check.
predicted_kinks = None          # e.g. [.., .., ..]

A_new = [3.0, 2.9, 2.8]
actual_kinks = [w * a for a in A_new]
check("kink locations", predicted_kinks, actual_kinks)

grid2, gains2 = sweep(A_new, chi, w)
plt.figure(figsize=(7, 3.6))
plt.plot(grid2, 100*gains2, lw=2); plt.gca().invert_xaxis()
plt.xlabel("r"); plt.ylabel("gain, %"); plt.grid(alpha=.3)
plt.title("Three similar tasks: the kinks bunch together"); plt.show()

---
## Now advise a real firm, with an agent

Everything so far had three tasks and you could check it by hand. Below
are two Norwegian firms with **eight tasks each**. Answering "who gains
more from cheaper AI" means running the model sixteen times and comparing
share-weighted sums. That is where hand-calculation stops being sensible,
and it is exactly the kind of question a manager asks.

So: give your `automation()` model to an agent as a **tool**, and ask in
English. The model does no arithmetic. It decides which runs it needs,
your code executes them, and the numbers come back.

In [ ]:
import json

# Your model, as a function. (The TUTOR notebook has its own ask() that
# talks to the course tutor; this is a different function, and both can
# live in one notebook.) Works with a free Gemini key (Google AI Studio)
# Course route: an OpenRouter key running Mistral. One-time setup: key icon
# in the left sidebar -> add secret OPENROUTER_API_KEY, allow notebook access.
# (A free Gemini key under GOOGLE_API_KEY also works, as a fallback.)
# Never paste a key into a cell.

# ---- WHICH MODEL THIS STUDENT GETS --------------------------------------
# INSTRUCTOR: set EXPERIMENT below. "same" gives everyone MODEL_A.
# "split" sends a random half to MODEL_A and the other half to MODEL_B,
# assigned from the student id, so the same student always lands in the
# same arm however many times they re-run the notebook.
EXPERIMENT = "same"                                  # "same" or "split"
MODEL_A    = "mistralai/mistral-medium-3-5"          # strong at tool calling
MODEL_B    = "mistralai/mistral-small-2603"          # smaller, cheaper
GEMINI_MODEL = "gemini-3.6-flash"                    # fallback route only

ACTIVE_MODEL = MODEL_A

def assign_model(student_id=""):
    """Pick this student's model. Deterministic: same id -> same arm."""
    global ACTIVE_MODEL
    if EXPERIMENT == "split" and student_id.strip():
        import hashlib
        digest = hashlib.sha256(student_id.strip().lower().encode()).hexdigest()
        ACTIVE_MODEL = MODEL_A if int(digest, 16) % 2 == 0 else MODEL_B
    else:
        ACTIVE_MODEL = MODEL_A
    return ACTIVE_MODEL

def _get_secret(name):
    """Colab Secrets first; an environment variable as the fallback, so the
    instructor can test outside Colab. Never a value pasted into a cell."""
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v:
            return v
    except Exception:
        pass
    import os
    return os.environ.get(name)

def _credentials():
    for secret, base, model in [
        ("OPENROUTER_API_KEY",
         "https://openrouter.ai/api/v1",
         ACTIVE_MODEL),
        ("GOOGLE_API_KEY",
         "https://generativelanguage.googleapis.com/v1beta/openai",
         GEMINI_MODEL),
    ]:
        key = _get_secret(secret)
        if key:
            return base, key, model
    return None

def _chat(messages, tools=None):
    """One HTTP call to the model. Returns its reply message, or None."""
    creds = _credentials()
    if creds is None:
        print("(no API key found -- see the key guide on It's Learning)")
        return None
    base, key, model = creds
    payload = {"model": model, "messages": messages}
    if tools:
        payload["tools"] = tools
        payload["tool_choice"] = "auto"
    try:
        import requests
        r = requests.post(base + "/chat/completions",
                          headers={"Authorization": "Bearer " + key},
                          json=payload, timeout=90)
        r.raise_for_status()
        return r.json()["choices"][0]["message"]
    except Exception as e:
        print(f"(the call failed: {type(e).__name__}: {e})")
        return None


def ask_model(prompt, system=None):
    """One plain call: text in, text out. No tools, no loop."""
    messages = ([{"role": "system", "content": system}] if system else [])
    messages.append({"role": "user", "content": prompt})
    reply = _chat(messages)
    return None if reply is None else reply.get("content")


# ---- THE AGENT ----------------------------------------------------------
# A model on its own only writes text. An agent is a model plus TOOLS plus
# a LOOP. Read Conversation.say() once: it is the whole idea, in 20 lines.

class Conversation:
    """An agent you can talk to. It remembers the exchange, and it can call
    your Python functions in the middle of answering -- or ask you a
    question first and wait for your reply."""

    def __init__(self, tools=None, registry=None, system=None, show=True):
        self.tools, self.registry = tools, registry or {}
        self.show = show
        self.messages = [{"role": "system", "content": system}] if system else []

    def say(self, text, max_steps=6):
        """Say something to the agent. Returns its reply, or None with no key."""
        self.messages.append({"role": "user", "content": text})
        for _ in range(max_steps):
            reply = _chat(self.messages, self.tools)
            if reply is None:
                return None
            self.messages.append(reply)
            calls = reply.get("tool_calls")
            if not calls:                       # no tool wanted: it is answering
                return reply.get("content")
            for call in calls:                  # it asked; WE run the function
                name = call["function"]["name"]
                args = json.loads(call["function"]["arguments"] or "{}")
                result = self.registry[name](**args)
                if self.show:
                    print(f"   [agent ran {name}({args})]")
                self.messages.append({"role": "tool", "tool_call_id": call["id"],
                                      "content": json.dumps(result)})
        return "(gave up: too many steps)"


def run_agent(question, tools, registry, max_steps=6, show=True):
    """One-shot version: ask once, get the answer. A Conversation of length 1."""
    return Conversation(tools, registry, show=show).say(question, max_steps)


# ---- THIS SESSION'S TUTOR ----------------------------------------------
# The rules are the same in every lab. What changes is the MODEL the agent
# is teaching and the PROBLEM SET it can look up -- both handed in below.

TUTOR_RULES = """You are a teaching assistant in a first-year microeconomics
course. Rules you always follow:
1. Explain the METHOD before any numbers, in the order the course teaches it.
2. Never invent numbers. Get them by calling your tools.
3. If a parameter you need has not been given, ASK for it and stop. Do not
   assume a value.
4. When you report a result, say in plain words what it means economically,
   including what a high and a low value of the key parameter would imply.
5. Asked about a problem set question, call get_problem FIRST so you work from
   the real wording. Explain the method and the setup. Do NOT give the final
   numbers: leave those to the student. If they show you an answer, check it
   with your tools and say only whether it is right and which step failed.
6. Asked to test the student, ask ONE question at a time and wait. Say whether
   the reply is right before asking the next. Test understanding rather than
   recall: ask why something holds, or what would change if a parameter moved.
7. Be brief. Six sentences at most."""


def make_tutor(session, model_text, problems, tools, registry, show=True):
    """Build this session's tutor: shared rules, this session's knowledge."""
    def get_problem(problem_id):
        return problems.get(problem_id,
                            "no problem " + str(problem_id) + " in this session")

    reg = dict(registry)
    reg["get_problem"] = get_problem
    tls = list(tools) + [{
        "type": "function",
        "function": {
            "name": "get_problem",
            "description": ("Fetch the exact wording of a problem from THIS "
                            "session's problem set, so you work from the real "
                            "question rather than one you imagined. Valid ids: "
                            + ", ".join(sorted(problems)) + "."),
            "parameters": {"type": "object", "properties": {
                "problem_id": {"type": "string", "enum": sorted(problems)}},
                "required": ["problem_id"]}}}]

    system = (TUTOR_RULES + """

THE MODEL YOU ARE TEACHING IN THIS SESSION:
""" + model_text + """

The student is working through a lab on exactly this material, and has the
problem set open beside them.""")
    return Conversation(tls, reg, system=system, show=show)


print("ask_model() and run_agent() ready")

### The two firms, and the tool

Read the tool description carefully. It is all the model knows about your
model: get it wrong and the agent reasons confidently about the wrong
thing.

In [ ]:
FIRMS = {
    "nordvik": {   # logistics: lots of automatable tasks, small cost shares
        "A":   [3.0, 2.8, 2.5, 2.2, 1.8, 1.2, 0.8, 0.4],
        "chi": [.05, .05, .05, .05, .10, .20, .25, .25]},
    "bergen": {    # consulting: few automatable tasks, but they are the expensive ones
        "A":   [2.6, 2.4, 1.0, 0.9, 0.8, 0.7, 0.6, 0.5],
        "chi": [.30, .25, .10, .08, .07, .07, .07, .06]},
}

def evaluate_firm(firm, r, w=1.0):
    """The bridge: the agent names a firm and a price of AI; YOUR model runs."""
    f = FIRMS[firm]
    out = automation(f["A"], f["chi"], w, r)
    auto = out["automated"]
    return {"firm": firm, "r": r, "cutoff_Abar": out["Abar"],
            "tasks_automated": [i + 1 for i, a in enumerate(auto) if a],
            "automated_cost_share": round(float(np.array(f["chi"])[auto].sum()), 4),
            "productivity_gain": round(out["dlnTFP"], 4)}

TOOLS = [{
    "type": "function",
    "function": {
        "name": "evaluate_firm",
        "description": ("Run the task model of automation for one firm at a given "
                        "AI rental rate r. A task is automated when its machine "
                        "productivity exceeds r/w. Returns which tasks are automated, "
                        "the share of total cost they represent, and the firm's "
                        "productivity gain."),
        "parameters": {
            "type": "object",
            "properties": {
                "firm": {"type": "string", "enum": ["nordvik", "bergen"]},
                "r": {"type": "number", "description": "rental rate of the machine, the price of AI"},
            },
            "required": ["firm", "r"],
        },
    },
}]

REGISTRY = {"evaluate_firm": evaluate_firm}

#@title Session 8 knowledge (double-click if you want to read it)
SESSION8_MODEL = """The task model of automation (Acemoglu). Production is a
set of TASKS, not one black box. Task i can be done by a worker at cost w, or
by a machine at cost r/A_i where A_i is the machine's productivity in that
task. The firm automates task i exactly when r/A_i < w, i.e. when
A_i > r/w = Abar, the cutoff. The cost saving on an automated task is
pi_i = 1 - Abar/A_i, and by Hulten's theorem the firm's productivity gain is
the COST-SHARE-WEIGHTED sum, sum_i chi_i * pi_i, where chi_i is task i's share
of total cost. Two consequences students underrate. First, cost shares matter
more than counts: automating many cheap tasks can be worth less than
automating one expensive one. Second, as r falls the gain is KINKED, not
smooth: each task crosses the cutoff one at a time, and a task that has only
just crossed saves almost nothing, so the extensive margin adds little on
impact. Session 8 also covers the firm's cost minimisation behind this."""

PROBLEMS8 = {
    "A8.1": ("Nordvik Marine builds aluminium hulls with F(E,K) = E^(1/2) "
             "K^(1/2), output price p = 50, wage w = 10, capital rental "
             "r = 10. Parts cover the firm's optimal input choice, returns to "
             "scale, and what happens to labour demand when the price of "
             "capital falls."),
    "A8.2": ("Fjordline Furniture: a task-model problem. Given machine "
             "productivities and cost shares across tasks, find which tasks "
             "are automated at a given r, compute the productivity gain as the "
             "share-weighted sum, and analyse how the answer moves as r falls."),
}

tutor8 = make_tutor(8, SESSION8_MODEL, PROBLEMS8, TOOLS, REGISTRY)
print(evaluate_firm("nordvik", 2.0))

### Question 1: who gains more, and why?

Predict first. Nordvik automates more tasks than Bergen at any price, so
the obvious guess is Nordvik.

In [ ]:
your_guess = None      # <-- YOUR TURN: "nordvik" or "bergen"

q1 = """The price of AI falls from r = 2.0 to r = 1.2, with the wage at 1.
Compare Nordvik and Bergen. Which firm gains more, by how much, and what is
the economic reason? Answer in at most four sentences for a manager."""

print(run_agent(q1, TOOLS, REGISTRY) or "(no key: run the four cases by hand below)")
print()
check("who gains more", your_guess, "bergen")

Bergen automates **two** tasks and Nordvik **five**, yet Bergen gains
28.7% against Nordvik's 14.1%. Counting automatable tasks tells you almost
nothing; those two tasks carry 55% of Bergen's costs. This is section 3's
lesson again, now as a business answer rather than an exercise.

### Question 2: a question with no closed form

Now something you cannot answer in one run, and neither can the agent. It
has to search: call the tool, read the result, adjust, call again. Watch
the `[agent ran ...]` lines and you are watching the loop work.

In [ ]:
q2 = """For Nordvik, with the wage at 1: how far does the price of AI have to
fall before at least half of the firm's total cost base is automated? Give the
threshold as precisely as you can, and say which task tips it over."""

print(run_agent(q2, TOOLS, REGISTRY) or "(no key: use the sweep below instead)")

### Check it yourself

Never take the last cell's word for it. One line of your own code settles
whether the agent's threshold is right.

In [ ]:
# <-- YOUR TURN: the r at which Nordvik's automated cost share first reaches 0.5.
agent_threshold = None      # what did the agent say?

rs = np.linspace(2.0, 0.5, 300)
shares = [evaluate_firm("nordvik", float(r))["automated_cost_share"] for r in rs]
first = next(r for r, s in zip(rs, shares) if s >= 0.5)
print(f"your model says the half-way point is r = {first:.3f}")
print("(the task with A = 1.2 is the one that tips it: 20% of costs in one task)")
check("agent's threshold", agent_threshold, first, tol=0.05)

plt.figure(figsize=(6.4, 3.4))
plt.plot(rs, shares, lw=2); plt.axhline(0.5, ls=":", color="red")
plt.gca().invert_xaxis(); plt.xlabel("r (falling ->)")
plt.ylabel("automated share of costs"); plt.grid(alpha=.3); plt.show()

### Let it teach the problem set, and examine you

Same agent, now asked to teach rather than advise. It carries this
session's model and can look up A8.1 and A8.2.

In [ ]:
print(tutor8.say("Walk me through how to approach A8.2. What is the method?")
      or "(no key)")

In [ ]:
print(tutor8.say("Now ask me one question to check I understood why cost "
                 "shares matter more than the number of automated tasks.")
      or "(no key)")

In [ ]:
my_reply = ""      # <-- YOUR TURN: answer in your own words
print(tutor8.say(my_reply) if my_reply else "answer the question above first")

---
## Take away

* One inequality, $A_i > r/w$, generates the whole model.
* The gain is *share-weighted*: a task nobody spends money on can be fully
  automated and change nothing. Bergen beats Nordvik on two tasks.
* The curve is kinked because tasks enter one at a time.
* An agent is your model plus a loop. It answered a question that needed
  sixteen model runs and a search, and every number in its answer came
  out of code you wrote. Change the model and the advice changes with it.